# Missing Backtests for ScArlet-Sails Paper (Track C)

Filling the 5 gaps in our dataset before paper writeup:

1. **Trend-following on crypto** (200d SMA on BTC/ETH/SOL × 4 TF) — we had trend only on metals.
2. **Mean-reversion on metals × multiple TF** (1h not just 1d) — we had only daily metals.
3. **Deflated Sharpe Ratio** (Bailey & Lopez de Prado 2014) on all results — academic gold standard.
4. **Probability of Backtest Overfitting (PBO)** per strategy class — second academic must-have.
5. **Cost sensitivity sweep** (1×/1.5×/2× bps) — shows where edge breaks.

**Setup**: re-clone repo, install deps, fetch both crypto and metals data, then run cells below.

**Output**: `paper/results/` JSON tables ready for paper tables/figures.

## 0. Setup

In [ ]:
%cd /kaggle/working
!rm -rf ScArlet-Sails
!git clone --branch claude/quizzical-raman-434cfb --depth 1 https://github.com/StarDust1508/ScArlet-Sails.git
!pip install -q -r ScArlet-Sails/requirements.txt 2>&1 | tail -3

In [ ]:
%cd /kaggle/working/ScArlet-Sails
# Fetch metals (always quick, ~30s)
!python scripts/fetch_metals.py 2>&1 | tail -5
# Fetch crypto (longer, ~10-15 min for 14 coins × 4 TF)
!python scripts/fetch_binance_klines.py --start 2023-01 --workers 12 2>&1 | tail -5
!ls data/raw/*.parquet | wc -l

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ScArlet-Sails')
sys.path.insert(0, '/kaggle/working/ScArlet-Sails/paper/notebooks')

import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import vectorbt as vbt

from core.data_loader import load_market_data
from strategies.simple_strategies import CombinedStrategy, SimpleRSIStrategy
from core.ood_detector import OODDetector
from stats import sharpe_ratio, deflated_sharpe, pbo, evaluate_strategy

RESULTS_DIR = Path('/kaggle/working/ScArlet-Sails/paper/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Trend-following on crypto (200d SMA on BTC/ETH/SOL × 4 TF)

Hypothesis: trend-following works on crypto same as metals, just with shorter history.

**Note on bars-per-year**: crypto trades 24/7 → 365 days. 15m → 35040 bars/year; 1h → 8760; 4h → 2190; 1d → 365.

In [ ]:
TF_BPY = {'15m': 35040, '1h': 8760, '4h': 2190, '1d': 365}
TF_FREQ = {'15m': '15min', '1h': '1h', '4h': '4h', '1d': '1D'}

def sma_trend_backtest(asset, tf, sma_period=200, fees=0.001, slippage=0.0005, capital=10_000):
    df = load_market_data(asset, tf, start_date='2023-01-01')
    if len(df) < sma_period + 50:
        return None
    sma = df['close'].rolling(sma_period).mean()
    above = df['close'] > sma
    entries = above & ~above.shift(1).fillna(False)
    exits = ~above & above.shift(1).fillna(False)
    pf = vbt.Portfolio.from_signals(
        close=df['close'], entries=entries, exits=exits,
        init_cash=capital, fees=fees, slippage=slippage,
        size=0.95, size_type='percent', freq=TF_FREQ[tf],
    )
    bh_ret = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    return {
        'asset': asset, 'tf': tf,
        'bars': len(df),
        'strategy_ret_pct': round(pf.total_return() * 100, 2),
        'sharpe': round(pf.sharpe_ratio(), 3),
        'maxDD_pct': round(pf.max_drawdown() * 100, 2),
        'trades': int(pf.trades.count()),
        'bh_ret_pct': round(bh_ret, 2),
        'edge_pct': round(pf.total_return() * 100 - bh_ret, 2),
    }

results_crypto_trend = []
for asset in ['BTC', 'ETH', 'SOL']:
    for tf in ['15m', '1h', '4h', '1d']:
        r = sma_trend_backtest(asset, tf)
        if r is not None:
            results_crypto_trend.append(r)
            print(f"{asset}/{tf}: ret={r['strategy_ret_pct']:+.1f}% sharpe={r['sharpe']:+.2f} trades={r['trades']}")

df_crypto_trend = pd.DataFrame(results_crypto_trend)
df_crypto_trend.to_json(RESULTS_DIR / 'crypto_trend_sma200.json', orient='records', indent=2)
print()
print(df_crypto_trend.to_string())
print(f"\nAvg Sharpe: {df_crypto_trend['sharpe'].mean():+.2f}")
print(f"Positive Sharpe: {(df_crypto_trend['sharpe'] > 0).sum()}/{len(df_crypto_trend)}")
print(f"Better than B&H: {(df_crypto_trend['edge_pct'] > 0).sum()}/{len(df_crypto_trend)}")

## 2. Mean-reversion on metals — multi-TF (1h not just 1d)

yfinance даёт 1h history only ~730 days. Тестируем что есть.

In [ ]:
# Fetch 1h metals (~2 years of intraday data)
!python /kaggle/working/ScArlet-Sails/scripts/fetch_metals.py --timeframes 1h 2>&1 | tail -8

import importlib, core.data_loader
importlib.reload(core.data_loader)
from core.data_loader import load_market_data

def combined_backtest(asset, tf, fees=0.0005, slippage=0.0002, capital=10_000):
    df = load_market_data(asset, tf)
    if len(df) < 500:
        return None
    strat = CombinedStrategy(ood_detector=OODDetector())
    sig = strat.generate_signals(df).reindex(df.index).fillna(0).astype(int)
    if (sig == 1).sum() < 2:
        return None
    pf = vbt.Portfolio.from_signals(
        close=df['close'], entries=sig==1, exits=sig==-1,
        init_cash=capital, fees=fees, slippage=slippage,
        size=0.95, size_type='percent', freq=TF_FREQ[tf],
    )
    bh_ret = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    return {
        'asset': asset, 'tf': tf, 'bars': len(df),
        'strategy_ret_pct': round(pf.total_return() * 100, 2),
        'sharpe': round(pf.sharpe_ratio(), 3),
        'maxDD_pct': round(pf.max_drawdown() * 100, 2),
        'trades': int(pf.trades.count()),
        'bh_ret_pct': round(bh_ret, 2),
        'edge_pct': round(pf.total_return() * 100 - bh_ret, 2),
    }

results_metals_multi_tf = []
for asset in ['GOLD', 'SILVER', 'COPPER', 'PLATINUM']:
    for tf in ['1h', '1d']:
        r = combined_backtest(asset, tf)
        if r is not None:
            results_metals_multi_tf.append(r)
            print(f"{asset}/{tf}: ret={r['strategy_ret_pct']:+.1f}% sharpe={r['sharpe']:+.2f} trades={r['trades']}")

df_metals_multi = pd.DataFrame(results_metals_multi_tf)
df_metals_multi.to_json(RESULTS_DIR / 'metals_combined_multi_tf.json', orient='records', indent=2)
print()
print(df_metals_multi.to_string())

## 3. Deflated Sharpe Ratio on all results

Bailey & Lopez de Prado (2014): корректирует Sharpe на:
- Selection bias (мы тестировали много стратегий)
- Non-normality (skew, kurt)

`n_trials_estimated=100` reflects honest count of strategy×param×asset combos we explored across the project.

Output: для каждого пары (asset, tf, strategy) — raw vs deflated Sharpe.

In [ ]:
# Re-run combined strategy on full universe, extract returns series, compute Deflated SR
all_returns = {}
for asset in ['BTC', 'ETH', 'SOL', 'GOLD', 'SILVER', 'COPPER', 'PLATINUM']:
    for tf in ['4h', '1d']:
        try:
            df = load_market_data(asset, tf, start_date='2023-01-01' if asset in ['BTC','ETH','SOL'] else None)
        except Exception:
            continue
        if len(df) < 500:
            continue
        strat = CombinedStrategy(ood_detector=OODDetector())
        sig = strat.generate_signals(df).reindex(df.index).fillna(0).astype(int)
        if (sig == 1).sum() < 5:
            continue
        pf = vbt.Portfolio.from_signals(
            close=df['close'], entries=sig==1, exits=sig==-1,
            init_cash=10_000, fees=0.0005, slippage=0.0002,
            size=0.95, size_type='percent', freq=TF_FREQ[tf],
        )
        returns = pf.returns().dropna()
        all_returns[f"{asset}_{tf}"] = returns

# Compute deflated Sharpe per series
deflated_results = []
for label, r in all_returns.items():
    bpy = TF_BPY[label.split('_')[-1]]
    metrics = evaluate_strategy(r, n_trials_estimated=100, periods_per_year=bpy, label=label)
    deflated_results.append(metrics)

df_deflated = pd.DataFrame(deflated_results)
df_deflated.to_json(RESULTS_DIR / 'deflated_sharpe.json', orient='records', indent=2)
print(df_deflated.to_string())
print()
print(f"Median raw Sharpe:       {df_deflated['sharpe_raw'].median():+.2f}")
print(f"Median deflated Sharpe:  {df_deflated['sharpe_deflated'].median():+.2f}")
print(f"Deflation ratio:         {(df_deflated['sharpe_deflated'] / df_deflated['sharpe_raw'].replace(0, 1)).median():.2f}")

## 4. Probability of Backtest Overfitting (PBO)

Bailey/Borwein/Lopez de Prado/Zhu (2014): combinatorial CV that tests whether the in-sample ranking of strategies survives out-of-sample.

**PBO > 0.5 = overfitting present.** Best in-sample strategies are essentially random when applied OOS.

In [ ]:
# Build returns matrix: rows = time, columns = strategies
# Use 4h crypto + 1d metals (consistent rebal cadence)
import pandas as pd

returns_matrix = pd.DataFrame(all_returns)
# Resample to daily for consistent comparison
returns_matrix = returns_matrix.resample('1D').sum().dropna(how='all')
print(f"Returns matrix shape: {returns_matrix.shape}")
print(f"Strategies: {list(returns_matrix.columns)}")

pbo_score, pbo_details = pbo(returns_matrix, n_splits=16)
print()
print(f"PBO score: {pbo_score:.3f}")
print(f"Details: {pbo_details}")
print()
if pbo_score > 0.5:
    print("INTERPRETATION: HIGH overfitting. Best in-sample strategy ≈ random OOS.")
elif pbo_score > 0.3:
    print("INTERPRETATION: MODERATE overfitting. Some selection bias present.")
else:
    print("INTERPRETATION: LOW overfitting. Ranking is reasonably stable.")

with open(RESULTS_DIR / 'pbo.json', 'w') as f:
    json.dump({'pbo_score': pbo_score, **pbo_details}, f, indent=2)

## 5. Cost sensitivity sweep

Бэктесты при разных предположениях о costs (1× / 1.5× / 2× базовый bps).
Показывает где edge ломается при реалистичной retail-стоимости.

In [ ]:
BASE_FEES = {'crypto': 0.001, 'metals': 0.0005}
BASE_SLIPPAGE = {'crypto': 0.0005, 'metals': 0.0002}

def cost_sensitivity(asset, tf, asset_class, multipliers=[1.0, 1.5, 2.0]):
    df = load_market_data(asset, tf, start_date='2023-01-01' if asset_class == 'crypto' else None)
    if len(df) < 500:
        return None
    strat = CombinedStrategy(ood_detector=OODDetector())
    sig = strat.generate_signals(df).reindex(df.index).fillna(0).astype(int)
    base_fees = BASE_FEES[asset_class]
    base_slip = BASE_SLIPPAGE[asset_class]
    out = {'asset': asset, 'tf': tf}
    for mult in multipliers:
        pf = vbt.Portfolio.from_signals(
            close=df['close'], entries=sig==1, exits=sig==-1,
            init_cash=10_000, fees=base_fees * mult, slippage=base_slip * mult,
            size=0.95, size_type='percent', freq=TF_FREQ[tf],
        )
        out[f'sharpe_{mult}x'] = round(pf.sharpe_ratio(), 3)
        out[f'ret_{mult}x_pct'] = round(pf.total_return() * 100, 2)
    return out

cost_results = []
for asset, ac in [('BTC', 'crypto'), ('ETH', 'crypto'), ('SOL', 'crypto'),
                  ('GOLD', 'metals'), ('SILVER', 'metals'), ('COPPER', 'metals'), ('PLATINUM', 'metals')]:
    for tf in ['4h', '1d']:
        try:
            r = cost_sensitivity(asset, tf, ac)
            if r is not None:
                cost_results.append(r)
        except Exception as e:
            print(f"{asset}/{tf}: {e}")

df_cost = pd.DataFrame(cost_results)
df_cost.to_json(RESULTS_DIR / 'cost_sensitivity.json', orient='records', indent=2)
print(df_cost.to_string())
print()
print('Sharpe degradation 1x → 2x:')
for _, row in df_cost.iterrows():
    delta = row['sharpe_2.0x'] - row['sharpe_1.0x']
    print(f"  {row['asset']}/{row['tf']}: Δ = {delta:+.2f}")

## 6. Aggregate summary table for paper

Combines all 5 analyses into one consolidated view.

In [ ]:
summary = {
    'crypto_trend_sma200': df_crypto_trend.to_dict('records'),
    'metals_combined_multi_tf': df_metals_multi.to_dict('records'),
    'deflated_sharpe': df_deflated.to_dict('records'),
    'pbo': {'score': pbo_score, **pbo_details},
    'cost_sensitivity': df_cost.to_dict('records'),
    'meta': {
        'date': pd.Timestamp.utcnow().isoformat(),
        'n_trials_estimated_for_DSR': 100,
        'crypto_period': '2023-01 to 2026-04',
        'metals_period': '1997-2026 (varies by metal)',
    }
}

with open(RESULTS_DIR / 'paper_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('=== PAPER SUMMARY ===')
print(f"\nCrypto SMA200 trend Avg Sharpe: {df_crypto_trend['sharpe'].mean():+.2f}")
print(f"Crypto SMA200 trend Positive:   {(df_crypto_trend['sharpe'] > 0).sum()}/{len(df_crypto_trend)}")
print(f"\nMetals multi-TF Avg Sharpe:     {df_metals_multi['sharpe'].mean():+.2f}")
print(f"\nDeflated Sharpe median:         {df_deflated['sharpe_deflated'].median():+.2f}")
print(f"  (raw median:                  {df_deflated['sharpe_raw'].median():+.2f})")
print(f"\nPBO score:                      {pbo_score:.3f}")
print(f"\nCost 1x→2x Sharpe degradation:")
df_cost['cost_delta'] = df_cost['sharpe_2.0x'] - df_cost['sharpe_1.0x']
print(f"  Median:                       {df_cost['cost_delta'].median():+.2f}")
print(f"\nAll results saved to {RESULTS_DIR}/")

## 7. What to look for in results

### Expected ranges (literature-based)

| Metric | Expected | What it means |
|---|---|---|
| Crypto SMA200 trend Avg Sharpe | **-0.5 to +0.5** | 2.5 years is short for trend; bull market most of period → SMA200 OK but few cycles |
| Metals multi-TF Sharpe | **0 to -0.5** | 1h on metals = too much noise for combined strategy |
| Deflated Sharpe median | **~50-70% of raw** | Standard correction. If <30% — extreme overfitting present |
| PBO score | **0.3-0.7** | Retail builds typically 0.5+ |
| Cost 1×→2× Sharpe delta | **-0.2 to -0.5** | Real-world cost variance kills edge |

### What WOULD be alarming

- Deflated < 30% of raw — strong evidence of overfitting
- PBO > 0.7 — best in-sample is systematically WORST out-of-sample
- Cost delta < -1.0 — any cost increase kills strategy

### Final use

All JSONs in `paper/results/` are inputs for paper figures and tables. The deflated Sharpe and PBO scores are **the academic credibility minimum** — without them, paper gets rejected at any peer-reviewed venue.